# Duka Electronics · Half-Year Sales Exploration (Jan–Jun 2026)
Prepared for the Head of Sales ahead of Thursday's review. Data: `duka_sales.csv` (1,000 orders, generated with seed 2026 by `generate_data.py`).

## Executive Summary
- Duka's 1,000 orders between 1 January and 29 June 2026 produced **KES 71.8M** in revenue (average order KES 71,792), and Nairobi alone delivered **36.4%** of it (KES 26.1M), almost double second-placed Mombasa at 19.5%.
- Laptops generate **57.9%** of revenue (KES 41.5M from 639 units), while Tablets sell more units (691) but earn only KES 19.3M, because a laptop costs KES 65,000 against KES 28,000.
- Weekend orders are **27.7%** of orders (277 orders, KES 19.9M) and are worth almost exactly the same as weekday orders (KES 71,972 vs KES 71,724, a 0.3% difference), so there is no weekend premium or discount.
- Revenue peaked in **April** at KES 13.5M and was lowest in **June** at KES 10.9M, so the best month earned **24.0%** more than the worst.
- The Premium online segment in Nairobi and Mombasa (the coast and capital cities) is **83 orders worth KES 10.2M**, 14.2% of total revenue, and is a ready-made target list for an online campaign.

## Setup

In [1]:
import numpy as np
import pandas as pd
import openpyxl

pd.set_option("display.width", 200)
pd.options.display.float_format = "{:,.1f}".format

## Task 1 · Load the data and the first-five-minutes inspection
The file is loaded with `parse_dates=["order_date"]` so that `order_date` is a true datetime from the start (this makes the `.dt` accessor work later).

In [2]:
sales = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
sales.head()

,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


In [3]:
sales.tail()

,order_id,order_date,branch,product,quantity,channel,unit_price
995,1996,2026-03-04,Mombasa,Laptop,5,In-store,65000
996,1997,2026-03-06,Nairobi,Tablet,2,Phone order,28000
997,1998,2026-01-20,Nairobi,Laptop,3,Online,65000
998,1999,2026-03-02,Nairobi,Headphones,2,In-store,2500
999,2000,2026-01-30,Nakuru,Charger,4,In-store,1200


In [4]:
print("shape:", sales.shape)
sales.info()

shape: (1000, 7)
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[us]
 2   branch      1000 non-null   str           
 3   product     1000 non-null   str           
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   str           
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[us](1), int64(3), str(3)
memory usage: 54.8 KB


In [5]:
sales.describe()

,order_id,order_date,quantity,unit_price
count,"1,000.0",1000,"1,000.0","1,000.0"
mean,"1,500.5",2026-03-30 08:44:09.600000,3.0,"23,918.3"
min,"1,001.0",2026-01-01 00:00:00,1.0,"1,200.0"
25%,"1,250.8",2026-02-14 00:00:00,2.0,"2,500.0"
50%,"1,500.5",2026-03-29 00:00:00,3.0,"15,000.0"
75%,"1,750.2",2026-05-14 00:00:00,4.0,"28,000.0"
max,"2,000.0",2026-06-29 00:00:00,5.0,"65,000.0"
std,288.8,NaN,1.4,"23,908.8"


In [6]:
print("distinct values per column:")
print(sales.nunique())
print("\nmissing values per column:"); print(sales.isna().sum())
print("\nfully duplicated rows:", sales.duplicated().sum(), "| duplicated order_id:", sales["order_id"].duplicated().sum())

distinct values per column:
order_id      1000
order_date     179
branch           5
product          5
quantity         5
channel          3
unit_price       5
dtype: int64

missing values per column:
order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64

fully duplicated rows: 0 | duplicated order_id: 0


In [7]:
for col in ["branch", "channel", "product"]:
    print(sales[col].value_counts(normalize=True).round(3).mul(100).rename(f"{col} %").to_string(), "\n")
print("Earliest order:", sales["order_date"].min().date(), "| latest order:", sales["order_date"].max().date())

branch
Nairobi   37.6
Mombasa   18.6
Nakuru    16.2
Kisumu    15.9
Eldoret   11.7 

channel
In-store      57.9
Online        32.0
Phone order   10.1 

product
Laptop       22.0
Tablet       21.5
Phone        19.4
Headphones   18.7
Charger      18.4 

Earliest order: 2026-01-01 | latest order: 2026-06-29


**Observations**
- **Shape:** 1,000 rows × 6 columns, one row per order (`order_id` runs 1001 to 2000, no repeats), so this is clean order-level data.
- **`.head()` / `.tail()`:** each order is for a single product and a single branch/channel. Rows are in `order_id` order, not date order, so dates should not be read as a timeline.
- **Types (`.info()`):** `order_date` is `datetime64` thanks to `parse_dates`; `quantity` and `unit_price` are integers; `branch`, `product` and `channel` are text (`object`/`str`). All six columns have 1,000 non-null values, so **there are no missing values and no duplicate rows**.
- **`.describe()`:** quantity runs from 1 to 5 (mean 3.0); unit price runs from KES 1,200 to KES 65,000 (mean KES 23,918), so prices span a factor of about 54 and will dominate revenue.
- **Distinct values:** 5 branches, 5 products, 3 channels, 179 distinct order dates.
- **Business mix:** Nairobi takes 37.6% of orders and the other four branches 12–19% each; 57.9% of orders are in-store, 32.0% online and 10.1% by phone; the five products are sold in near-equal numbers of orders (18–22% each).
- **Period covered:** 01 January 2026 to 29 June 2026, i.e. 180 calendar days (the half-year ends on 30 June, and the last order in the file is 29 June).

## Task 2 · Derived columns
`revenue`, `month` (month name), `weekday` (day name), `is_weekend` (Saturday/Sunday) and `price_band` (Premium at or above KES 28,000, otherwise Value).

In [8]:
sales["revenue"] = sales["quantity"] * sales["unit_price"]
sales["month"] = sales["order_date"].dt.month_name()
sales["weekday"] = sales["order_date"].dt.day_name()
sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5
sales["price_band"] = np.where(sales["unit_price"] >= 28000, "Premium", "Value")

month_order = ["January", "February", "March", "April", "May", "June"]
sales[["order_id", "order_date", "quantity", "unit_price", "revenue", "month", "weekday", "is_weekend", "price_band"]].head()

,order_id,order_date,quantity,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,3,15000,45000,June,Wednesday,False,Value
1,1002,2026-02-02,4,65000,260000,February,Monday,False,Premium
2,1003,2026-01-05,5,28000,140000,January,Monday,False,Premium
3,1004,2026-04-26,5,2500,12500,April,Sunday,True,Value
4,1005,2026-03-07,3,2500,7500,March,Saturday,True,Value


In [9]:
for col in ["month", "weekday", "is_weekend", "price_band"]:
    print(sales[col].value_counts().to_string(), "\n")
print("total revenue:", f"KES {sales['revenue'].sum():,.0f}")

month
March       186
January     173
April       167
June        162
February    159
May         153 

weekday
Friday       166
Sunday       145
Thursday     145
Monday       140
Wednesday    137
Tuesday      135
Saturday     132 

is_weekend
False    723
True     277 

price_band
Value      565
Premium    435 

total revenue: KES 71,792,500


**Sanity checks.** Revenue totals **KES 71,792,500**; every month January–June appears; all seven weekdays appear; `is_weekend` splits into 277 weekend and 723 weekday orders; `price_band` gives 435 Premium orders (Tablets at KES 28,000 and Laptops at KES 65,000 both meet the threshold) and 565 Value orders (Phones, Headphones, Chargers). The values look right, so the columns can be trusted.

## Task 3 · Branches ranked by revenue and by number of orders

In [10]:
branch = (sales.groupby("branch")
               .agg(total_revenue=("revenue", "sum"), orders=("order_id", "count"))
               .assign(share_pct=lambda t: t["total_revenue"] / t["total_revenue"].sum() * 100,
                       avg_order_value=lambda t: t["total_revenue"] / t["orders"]))
branch["rank_by_revenue"] = branch["total_revenue"].rank(ascending=False).astype(int)
branch["rank_by_orders"] = branch["orders"].rank(ascending=False).astype(int)
branch.sort_values("total_revenue", ascending=False)

,total_revenue,orders,share_pct,avg_order_value,rank_by_revenue,rank_by_orders
branch,,,,,,
Nairobi,26106100,376,36.4,"69,431.1",1,1
Mombasa,14003700,186,19.5,"75,288.7",2,2
Nakuru,11970400,162,16.7,"73,891.4",3,3
Kisumu,10868100,159,15.1,"68,352.8",4,4
Eldoret,8844200,117,12.3,"75,591.5",5,5


In [11]:
rev_rank = branch["total_revenue"].sort_values(ascending=False).index.tolist()
ord_rank = branch["orders"].sort_values(ascending=False).index.tolist()
print("by revenue:", rev_rank); print("by orders: ", ord_rank); print("same ranking:", rev_rank == ord_rank)

by revenue: ['Nairobi', 'Mombasa', 'Nakuru', 'Kisumu', 'Eldoret']
by orders:  ['Nairobi', 'Mombasa', 'Nakuru', 'Kisumu', 'Eldoret']
same ranking: True


**Answer: yes, the two rankings are identical** (Nairobi, Mombasa, Nakuru, Kisumu, Eldoret).

**Why.** Branch revenue is roughly *orders × average order value*. Orders differ hugely between branches (117 in Eldoret to 376 in Nairobi, a 3.2× spread), whereas average order value barely varies (KES 68,353 in Kisumu to KES 75,591 in Eldoret, only a 11% spread). Because volume differences are so much larger than basket-size differences, volume decides the revenue rank.

**Context and caution.**
- Nairobi has 36.4% of revenue and 37.6% of orders, 1.86× Mombasa.
- Eldoret has the *highest* average order (KES 75,591) but the fewest orders (117), so growth there is a volume problem, not a basket problem.
- Nakuru and Kisumu are nearly tied on orders (162 vs 159); Nakuru is KES 1,102,300 ahead on revenue only because its average order is about KES 5,539 larger. That 3rd/4th place is the least secure ranking.

## Task 4 · Top three products by revenue and by units sold

In [12]:
product = sales.groupby("product").agg(revenue=("revenue", "sum"), units=("quantity", "sum"), unit_price=("unit_price", "first"))
product["revenue_share_pct"] = product["revenue"] / product["revenue"].sum() * 100
print("Top 3 by revenue:"); print(product.nlargest(3, "revenue")[["revenue", "units", "unit_price", "revenue_share_pct"]], "\n")
print("Top 3 by units sold:"); print(product.nlargest(3, "units")[["units", "revenue", "unit_price"]])

Top 3 by revenue:
          revenue  units  unit_price  revenue_share_pct
product                                                
Laptop   41535000    639       65000               57.9
Tablet   19348000    691       28000               26.9
Phone     8880000    592       15000               12.4 

Top 3 by units sold:
         units   revenue  unit_price
product                             
Tablet     691  19348000       28000
Laptop     639  41535000       65000
Phone      592   8880000       15000


In [13]:
product.sort_values('revenue', ascending=False)

,revenue,units,unit_price,revenue_share_pct
product,,,,
Laptop,41535000,639,65000,57.9
Tablet,19348000,691,28000,26.9
Phone,8880000,592,15000,12.4
Headphones,1387500,555,2500,1.9
Charger,642000,535,1200,0.9


**Answer.** By revenue the top three are **Laptop (KES 41.5M), Tablet (KES 19.3M) and Phone (KES 8.9M)**. By units they are **Tablet (691 units), Laptop (639) and Phone (592)**.

**Difference.** The same three products appear in both lists, but Laptop and Tablet swap places. Tablets sold 52 more units than laptops (8% more) but earn less than half the revenue (47% of the laptop figure), because a laptop costs KES 65,000 and a tablet KES 28,000. Unit sales are similar across all five products (535 to 691), so the revenue ranking is almost entirely a *price* ranking. Headphones and chargers sell 555 and 535 units but together bring only 2.8% of revenue. **Implication:** laptop availability and pricing matter far more to revenue than the volume of any other product.

## Task 5 · Weekend vs weekday orders

In [14]:
wk = (sales.groupby("is_weekend")
           .agg(orders=("order_id", "count"), total_revenue=("revenue", "sum"), avg_order_revenue=("revenue", "mean"))
           .rename(index={False: "Weekday", True: "Weekend"}))
wk["order_share_pct"] = wk["orders"] / wk["orders"].sum() * 100
wk["revenue_share_pct"] = wk["total_revenue"] / wk["total_revenue"].sum() * 100
wk

,orders,total_revenue,avg_order_revenue,order_share_pct,revenue_share_pct
is_weekend,,,,,
Weekday,723,51856200,"71,723.7",72.3,72.2
Weekend,277,19936300,"71,972.2",27.7,27.8


In [15]:
all_days = pd.date_range(sales["order_date"].min(), sales["order_date"].max())
n_weekend_days = int((all_days.dayofweek >= 5).sum()); n_weekday_days = len(all_days) - n_weekend_days
print(f"calendar days in period: {n_weekday_days} weekdays, {n_weekend_days} weekend days")
print(f"orders per weekday: {wk.loc['Weekday','orders'] / n_weekday_days:.2f} | orders per weekend day: {wk.loc['Weekend','orders'] / n_weekend_days:.2f}")

calendar days in period: 128 weekdays, 52 weekend days
orders per weekday: 5.65 | orders per weekend day: 5.33


**Answer.** Weekdays: **723 orders, KES 51,856,200, average KES 71,724**. Weekends: **277 orders, KES 19,936,300, average KES 71,972**.

**Interpretation.** Weekdays account for 72.3% of orders and 72.2% of revenue simply because there are five of them against two weekend days. The fair comparison is per day: weekends see 5.33 orders per day against 5.65 on weekdays (fewer on weekends by 5.7%), and the average order is almost identical (0.3% apart). **Weekend days are slightly quieter than weekdays (5.7% fewer orders per day), but each order is just as valuable**, so revenue per day is only a few percent different. A gap this small could easily be chance in a six-month sample, so the evidence does not justify cutting weekend staffing or stock.

## Task 6 · Best and worst month for revenue

In [16]:
monthly = (sales.groupby("month")
                .agg(revenue=("revenue", "sum"), orders=("order_id", "count"), avg_order_value=("revenue", "mean"),
                     days_with_orders=("order_date", "nunique"))
                .reindex(month_order))
monthly

,revenue,orders,avg_order_value,days_with_orders
month,,,,
January,11786600,173,"68,130.6",31
February,11772000,159,"74,037.7",28
March,12396800,186,"66,649.5",31
April,13518300,167,"80,947.9",30
May,11415400,153,"74,610.5",31
June,10903400,162,"67,304.9",28


In [17]:
best_m, worst_m = monthly["revenue"].idxmax(), monthly["revenue"].idxmin()
best_v, worst_v = monthly["revenue"].max(), monthly["revenue"].min()
print(f"best:  {best_m}  KES {best_v:,.0f}")
print(f"worst: {worst_m}  KES {worst_v:,.0f}")
print(f"best is {(best_v / worst_v - 1) * 100:.1f}% higher than worst; worst is {(1 - worst_v / best_v) * 100:.1f}% lower than best")

best:  April  KES 13,518,300
worst: June  KES 10,903,400
best is 24.0% higher than worst; worst is 19.3% lower than best


**Answer.** The best month is **April (KES 13,518,300)** and the worst is **June (KES 10,903,400)**. April earned **24.0% more** than June (equivalently, June was 19.3% below April). The percentage difference is quoted relative to the worst month, the usual "uplift" convention.

**Context.** April did not win on volume: it had 167 orders against 186 in March, but a higher average order (KES 80,948 against KES 67,305 in June). So the peak came from bigger baskets (more laptop-heavy orders), not more customers. June data end on 29 June (29 of 30 days), which explains at most about 3% of the shortfall, so June is genuinely the weakest month, and revenue fell from the April peak to June. The monthly values should be read as a signal worth investigating, not proof of a seasonal trend, since the data cover only six months.

## Task 7 · Premium online orders from Nairobi and Mombasa

In [18]:
premium_online = sales[(sales["price_band"] == "Premium")
                       & (sales["channel"] == "Online")
                       & (sales["branch"].isin(["Nairobi", "Mombasa"]))].copy()

print("orders:", len(premium_online))
print(f"total revenue: KES {premium_online['revenue'].sum():,.0f}")
print(f"share of all revenue: {premium_online['revenue'].sum() / sales['revenue'].sum() * 100:.1f}%")
print(premium_online.groupby("branch")["revenue"].agg(["count", "sum"]), "\n")
print(premium_online.groupby("product")["revenue"].agg(["count", "sum"]))

orders: 83
total revenue: KES 10,218,000
share of all revenue: 14.2%
         count      sum
branch                 
Mombasa     33  3696000
Nairobi     50  6522000 

         count      sum
product                
Laptop      33  5850000
Tablet      50  4368000


In [19]:
premium_online.to_csv("outputs/premium_online_coast_capital.csv", index=False)
check = pd.read_csv("outputs/premium_online_coast_capital.csv")
print(check.shape, "| unnamed index column present:", any(c.startswith("Unnamed") for c in check.columns))

(83, 12) | unnamed index column present: False


**Answer.** There are **83 Premium online orders** from Nairobi and Mombasa, worth **KES 10,218,000** in total (14.2% of all revenue from 8.3% of orders). Nairobi accounts for 50 orders (KES 6,522,000) and Mombasa for 33 (KES 3,696,000). Laptops make up 57% of this segment's revenue from 33 orders; the other 50 orders are Tablets. The orders are saved to `outputs/premium_online_coast_capital.csv` (with `index=False`).

## Task 8 · Spreadsheet cross-check (`excel/duka_branch_summary.xlsx`)
The workbook has a **Sales** sheet with `revenue = quantity × unit_price`, a **Branches** sheet, an XLOOKUP region column, a **Branch Summary** (SUMIFS) and a **Pivot** sheet (revenue by branch and channel). The cell below reads the workbook's calculated values and compares them with pandas.

In [20]:
wb = openpyxl.load_workbook("excel/duka_branch_summary.xlsx", data_only=True)

# Branch ranking (SUMIFS) vs pandas
bs = wb["Branch Summary"]
xl = pd.DataFrame([[r[1], r[2], r[3], r[4]] for r in bs.iter_rows(min_row=2, max_row=6, values_only=True)],
                  columns=["branch", "region", "excel_revenue", "excel_orders"]).set_index("branch")
cmp1 = xl.join(branch[["total_revenue", "orders"]])
cmp1["revenue_matches"] = cmp1["excel_revenue"] == cmp1["total_revenue"]
cmp1["orders_match"] = cmp1["excel_orders"] == cmp1["orders"]
print(cmp1.to_string())

# Pivot (branch x channel) vs pandas
pv = wb["Pivot"]
xp = pd.DataFrame([list(r[1:4]) for r in pv.iter_rows(min_row=4, max_row=8, values_only=True)],
                  index=[r[0] for r in pv.iter_rows(min_row=4, max_row=8, values_only=True)],
                  columns=["In-store", "Online", "Phone order"])
pp = sales.pivot_table(index="branch", columns="channel", values="revenue", aggfunc="sum")
print("\nPivot matches pandas:", bool((xp.sort_index() == pp.sort_index()).all().all()))
print("Excel grand total:", f"{pv['E9'].value:,.0f}", "| pandas total:", f"{sales['revenue'].sum():,.0f}")
assert cmp1["revenue_matches"].all() and pv["E9"].value == sales["revenue"].sum()

              region  excel_revenue  excel_orders  total_revenue  orders  revenue_matches  orders_match
branch                                                                                                 
Nairobi      Central       26106100           376       26106100     376             True          True
Mombasa        Coast       14003700           186       14003700     186             True          True
Nakuru   Rift Valley       11970400           162       11970400     162             True          True
Kisumu       Western       10868100           159       10868100     159             True          True
Eldoret   North Rift        8844200           117        8844200     117             True          True



Pivot matches pandas: True
Excel grand total: 71,792,500 | pandas total: 71,792,500


**Confirmation.** The Excel SUMIFS branch totals, the order counts, the branch × channel cross-tab and the grand total (KES 71,792,500) all match pandas exactly, and the region column assigns Nairobi to Central, Mombasa to Coast, Kisumu to Western, Nakuru to Rift Valley and Eldoret to North Rift.

## Caveats
- Revenue is `quantity × unit_price`; the data has no discounts, costs or margins, so revenue is not profit.
- The data are synthetic and cover one half-year, so monthly differences are indicative only.
- The Executive Summary at the top of this notebook is the page for the Head of Sales; the sections above give the working behind each number.